
<h2 style="border:2px solid #AA5E7A;border-radius:100%; padding:20px; margin:0; color:#AA5E7A; text-align:center"> Data Understanding and Processing</h2>


In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

In [2]:
raw_path='../../data/raw/INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls'
df=pd.read_excel(raw_path,sheet_name=0)
defs=pd.read_excel(raw_path,sheet_name=1,header=None)
defs.to_csv('../../references/data_definitions.csv',index=False,header=False)
df.head()

,EmpNumber,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,...,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating
0,E1001000,32,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,10,3,...,4,10,2,2,10,7,0,8,No,3
1,E1001006,47,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,14,4,...,4,20,2,3,7,7,1,7,No,3
2,E1001007,40,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Frequently,5,4,...,3,20,2,3,18,13,1,12,No,4
3,E1001009,41,Male,Human Resources,Divorced,Human Resources,Manager,Travel_Rarely,10,4,...,2,23,2,2,21,6,12,6,No,3
4,E1001010,60,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,16,4,...,4,10,1,3,2,2,2,2,No,3


In [3]:
print('Shape (rows, columns):',df.shape)

Shape (rows, columns): (1200, 28)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   EmpNumber                     1200 non-null   object
 1   Age                           1200 non-null   int64 
 2   Gender                        1200 non-null   object
 3   EducationBackground           1200 non-null   object
 4   MaritalStatus                 1200 non-null   object
 5   EmpDepartment                 1200 non-null   object
 6   EmpJobRole                    1200 non-null   object
 7   BusinessTravelFrequency       1200 non-null   object
 8   DistanceFromHome              1200 non-null   int64 
 9   EmpEducationLevel             1200 non-null   int64 
 10  EmpEnvironmentSatisfaction    1200 non-null   int64 
 11  EmpHourlyRate                 1200 non-null   int64 
 12  EmpJobInvolvement             1200 non-null   int64 
 13  EmpJobLevel       

In [5]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Age,1200.0,36.92,9.09,18.0,30.0,36.0,43.0,60.0
DistanceFromHome,1200.0,9.17,8.18,1.0,2.0,7.0,14.0,29.0
EmpEducationLevel,1200.0,2.89,1.04,1.0,2.0,3.0,4.0,5.0
EmpEnvironmentSatisfaction,1200.0,2.72,1.09,1.0,2.0,3.0,4.0,4.0
EmpHourlyRate,1200.0,65.98,20.21,30.0,48.0,66.0,83.0,100.0
EmpJobInvolvement,1200.0,2.73,0.71,1.0,2.0,3.0,3.0,4.0
EmpJobLevel,1200.0,2.07,1.11,1.0,1.0,2.0,3.0,5.0
EmpJobSatisfaction,1200.0,2.73,1.10,1.0,2.0,3.0,4.0,4.0
NumCompaniesWorked,1200.0,2.66,2.47,0.0,1.0,2.0,4.0,9.0
EmpLastSalaryHikePercent,1200.0,15.22,3.63,11.0,12.0,14.0,18.0,25.0


In [6]:
df.describe(include=['object','string'])

,EmpNumber,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,OverTime,Attrition
count,1200,1200,1200,1200,1200,1200,1200,1200,1200
unique,1200,2,6,3,6,19,3,2,2
top,E1001000,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Rarely,No,No
freq,1,725,492,548,373,270,846,847,1022


### Missing values, duplicates and impossible values

In [7]:
print('Missing values:',df.isna().sum().sum())

Missing values: 0


In [8]:
print('Duplicate rows:',df.duplicated().sum())


Duplicate rows: 0


In [9]:
print('Duplicate EmpNumber:',df.EmpNumber.duplicated().sum())

Duplicate EmpNumber: 0


In [10]:
ordinal=['EmpEducationLevel','EmpEnvironmentSatisfaction','EmpJobInvolvement','EmpJobSatisfaction','EmpRelationshipSatisfaction','EmpWorkLifeBalance']
print(df[ordinal].agg(['min','max']))
print(df.PerformanceRating.value_counts().sort_index())

     EmpEducationLevel  EmpEnvironmentSatisfaction  EmpJobInvolvement  \
min                  1                           1                  1   
max                  5                           4                  4   

     EmpJobSatisfaction  EmpRelationshipSatisfaction  EmpWorkLifeBalance  
min                   1                            1                   1  
max                   4                            4                   4  
PerformanceRating
2    194
3    874
4    132
Name: count, dtype: int64


Finding: the data is already clean: no missing values, no duplicates and no impossible values. No rows were removed and nothing needed imputing. (If values had been missing, the plan was the median for numeric columns and the mode for categorical ones.)

### Outlier check (IQR rule)


In [11]:
num=df.select_dtypes('number').columns.drop('PerformanceRating')
q1,q3=df[num].quantile(.25),df[num].quantile(.75); iqr=q3-q1
out=((df[num]<q1-1.5*iqr)|(df[num]>q3+1.5*iqr)).sum()
out[out>0].sort_values(ascending=False).to_frame('IQR outliers')

,IQR outliers
TrainingTimesLastYear,188
YearsSinceLastPromotion,88
ExperienceYearsAtThisCompany,56
TotalWorkExperienceInYears,51
NumCompaniesWorked,39
ExperienceYearsInCurrentRole,16
YearsWithCurrManager,11


### Variable types
- Target: PerformanceRating (2, 3, 4). It has only three ordered values, so this is a classification task.
- Identifier: EmpNumber (dropped before modelling; it carries no information).
- Nominal categorical: Gender, EducationBackground, MaritalStatus, EmpDepartment, EmpJobRole, BusinessTravelFrequency, OverTime, Attrition.
- Ordinal scores (kept as numbers): education level, environment / job / relationship satisfaction, job involvement, work-life balance, job level.
- Continuous / count: age, distance, hourly rate, salary hike %, experience years, training times and others.

### Save the cleaned dataset

In [12]:
df.to_csv('../../data/processed/employee_clean.csv',index=False)
print(df.shape)

(1200, 28)


### Encoding and scaling (what is done and where)
Encoding and scaling are done inside a scikit-learn pipeline in train_model.ipynb, fitted on the training data only. This is deliberate: it avoids data leakage and means the same transformation is applied automatically at prediction time.

- Nominal columns -> one-hot encoding (OverTime, Attrition and Gender become a single 0/1 column).
- Numeric columns -> standard scaling (needed by Logistic Regression, KNN, SVM and the neural network; harmless for trees).

Below is a small preview of what the encoded data looks like, for reference only (the pipeline does not use this file).

In [13]:
cats=df.select_dtypes(['object','string']).columns.drop('EmpNumber')
preview=pd.get_dummies(df.drop(columns='EmpNumber'),columns=list(cats),drop_first=True,dtype=int)
preview.to_csv('../../data/processed/employee_encoded_preview.csv',index=False)
print(preview.shape)
preview.head(3)

(1200, 54)


,Age,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,EmpLastSalaryHikePercent,...,EmpJobRole_Sales Executive,EmpJobRole_Sales Representative,EmpJobRole_Senior Developer,EmpJobRole_Senior Manager R&D,EmpJobRole_Technical Architect,EmpJobRole_Technical Lead,BusinessTravelFrequency_Travel_Frequently,BusinessTravelFrequency_Travel_Rarely,OverTime_Yes,Attrition_Yes
0,32,10,3,4,55,3,2,4,1,12,...,1,0,0,0,0,0,0,1,0,0
1,47,14,4,4,42,3,2,1,2,12,...,1,0,0,0,0,0,0,1,0,0
2,40,5,4,4,48,2,3,1,5,21,...,1,0,0,0,0,0,1,0,1,0


### Summary
- 1200 employees, 28 columns (27 after dropping the ID); target PerformanceRating: 2 = about 16%, 3 = about 73%, 4 = about 11%.
- No cleaning losses.
- Saved: data/processed/employee_clean.csv (used by all later notebooks).